<a href="https://colab.research.google.com/github/Prktriesitout/LangGraph/blob/main/Project4Langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Agents that run code are risky if they run it without anyone checking it, This project builds a LangGraph workflow that pauses before executing a python expression , shows it to a human  and only continues if human answers yes.

Approved : The graph resumes, evaluates the expression and returns the result

Rejected : The graph skips execution and reports that .

START -> parse_expression -> human_approve -> yes -> execute_code -> END -> handle_reject -> END

In [1]:
%%writefile requirements.txt
langgraph>=0.4.0
langchain-core>=0.3.0

Overwriting requirements.txt


In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

# 1. State
class State(TypedDict):
    expression: str
    approval: str
    result: str

# 2. parse_expression: shows the expression
def parse_expression(state: State):
    print(f"Expression parsed: {state['expression']}")
    return {}

# 3. human_approve: pauses the graph and waits for yes / no
def human_approve(state: State):
    answer = interrupt({
        "question": "Approve execution? (yes / no)",
        "expression": state["expression"],
    })
    return {"approval": str(answer).strip().lower()}

# 4. execute_code: runs only on approval
def execute_code(state: State):
    print("Human approved execution.")
    # Restricted builtins: fine for a demo, but NOT a real sandbox
    value = eval(state["expression"], {"__builtins__": {}}, {})
    print(f"Result: {value}")
    return {"result": str(value)}

# 5. handle_reject: runs on rejection
def handle_reject(state: State):
    print("Human rejected execution.")
    print("Result: Execution skipped.")
    return {"result": "Execution skipped."}

# 6. Router for the conditional edge
def route_after_approval(state: State):
    return "execute_code" if state["approval"] == "yes" else "handle_reject"

In [4]:
# Build graph
builder = StateGraph(State)
builder.add_node("parse_expression", parse_expression)
builder.add_node("human_approve", human_approve)
builder.add_node("execute_code", execute_code)
builder.add_node("handle_reject", handle_reject)

builder.add_edge(START, "parse_expression")
builder.add_edge("parse_expression", "human_approve")
builder.add_conditional_edges(
    "human_approve",
    route_after_approval,
    {"execute_code": "execute_code", "handle_reject": "handle_reject"},
)
builder.add_edge("execute_code", END)
builder.add_edge("handle_reject", END)

# The checkpointer is required for interrupt/resume
graph = builder.compile(checkpointer=MemorySaver())

# Helper: run until the pause, then resume with the human's answer
def run_case(title, expression, human_answer, thread_id):
    print(f"\n===== {title} =====")
    config = {"configurable": {"thread_id": thread_id}}

    graph.invoke({"expression": expression, "approval": "", "result": ""}, config)
    print(f"[paused before: {graph.get_state(config).next}]")
    print(f"[human answers: {human_answer}]")

    graph.invoke(Command(resume=human_answer), config)

# Case 1: Approved
run_case("Case 1: Approved", "5 * (3 + 2)", "yes", thread_id="case-1")

# Case 2: Rejected
run_case("Case 2: Rejected", "open('file.txt', 'w')", "no", thread_id="case-2")


===== Case 1: Approved =====
Expression parsed: 5 * (3 + 2)
[paused before: ('human_approve',)]
[human answers: yes]
Human approved execution.
Result: 25

===== Case 2: Rejected =====
Expression parsed: open('file.txt', 'w')
[paused before: ('human_approve',)]
[human answers: no]
Human rejected execution.
Result: Execution skipped.


In [4]:
!pip install --upgrade --force-reinstall langchain-core langchain langchain-openai langgraph


  Using cached langchain_core-1.8.2-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain-1.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_openai-1.8.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached langgraph-1.2.22-py3-none-any.whl.metadata (4.9 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.47-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_protocol-0.0.21-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.15.0-py3-none-any.whl.metadata (40 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pydantic-2.14.3-py3-none-any.whl.metadata (141 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.meta

In [1]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI")

In [8]:
import re
import ast
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_openai import ChatOpenAI

# ---------- State ----------
class State(TypedDict):
    request: str
    expression: str
    approval: str
    result: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ---------- Helper: clean the LLM output ----------
def clean_expression(text: str) -> str:
    text = text.strip()
    fenced = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1)
    return text.strip()

# ---------- Nodes ----------
def generate_expression(state: State):
    """If an expression is already given, keep it. Otherwise ask the LLM for one."""
    if state["expression"]:
        return {}
    msg = llm.invoke(
        "Convert the request into a single Python expression. "
        "Reply with ONLY the expression, no backticks, no explanation.\n\n"
        f"Request: {state['request']}"
    )
    return {"expression": clean_expression(msg.content)}

def parse_expression(state: State):
    print(f"Expression parsed: {state['expression']}")
    return {}

def human_approve(state: State):
    answer = interrupt({
        "question": "Approve execution? (yes / no)",
        "expression": state["expression"],
    })
    return {"approval": str(answer).strip().lower()}

def execute_code(state: State):
    print("Human approved execution.")
    try:
        ast.parse(state["expression"], mode="eval")  # must be a single expression
        # Restricted builtins: fine for a demo, but NOT a real sandbox
        value = eval(state["expression"], {"__builtins__": {}}, {})
        print(f"Result: {value}")
        return {"result": str(value)}
    except Exception as e:
        print(f"Result: could not evaluate ({type(e).__name__}: {e})")
        return {"result": f"Error: {e}"}

def handle_reject(state: State):
    print("Human rejected execution.")
    print("Result: Execution skipped.")
    return {"result": "Execution skipped."}

# ---------- Router ----------
def route_after_approval(state: State):
    return "execute_code" if state["approval"] in ("yes", "y") else "handle_reject"

In [9]:
builder = StateGraph(State)
builder.add_node("generate_expression", generate_expression)
builder.add_node("parse_expression", parse_expression)
builder.add_node("human_approve", human_approve)
builder.add_node("execute_code", execute_code)
builder.add_node("handle_reject", handle_reject)

builder.add_edge(START, "generate_expression")
builder.add_edge("generate_expression", "parse_expression")
builder.add_edge("parse_expression", "human_approve")
builder.add_conditional_edges(
    "human_approve",
    route_after_approval,
    {"execute_code": "execute_code", "handle_reject": "handle_reject"},
)
builder.add_edge("execute_code", END)
builder.add_edge("handle_reject", END)

graph = builder.compile(checkpointer=MemorySaver())

In [10]:
def run_case(title, expression, human_answer, thread_id):
    print(f"\n===== {title} =====")
    config = {"configurable": {"thread_id": thread_id}}
    graph.invoke(
        {"request": "", "expression": expression, "approval": "", "result": ""},
        config,
    )
    print(f"[paused before: {graph.get_state(config).next}]")
    print(f"[human answers: {human_answer}]")
    graph.invoke(Command(resume=human_answer), config)

run_case("Case 1: Approved", "5 * (3 + 2)", "yes", thread_id="case-1")
run_case("Case 2: Rejected", "open('file.txt', 'w')", "no", thread_id="case-2")


===== Case 1: Approved =====
Expression parsed: 5 * (3 + 2)
[paused before: ('human_approve',)]
[human answers: yes]
Human approved execution.
Result: 25

===== Case 2: Rejected =====
Expression parsed: open('file.txt', 'w')
[paused before: ('human_approve',)]
[human answers: no]
Human rejected execution.
Result: Execution skipped.


In [11]:
def run_interactive(request, thread_id):
    print(f"\n===== Request: {request} =====")
    config = {"configurable": {"thread_id": thread_id}}
    graph.invoke(
        {"request": request, "expression": "", "approval": "", "result": ""},
        config,
    )
    if graph.get_state(config).next:  # paused at human_approve
        answer = input("Approve execution? (yes / no): ")
        graph.invoke(Command(resume=answer), config)

run_interactive("multiply 5 by the sum of 3 and 2", thread_id="live-1")
run_interactive("create a file called file.txt and open it for writing", thread_id="live-2")


===== Request: multiply 5 by the sum of 3 and 2 =====
Expression parsed: 5 * (3 + 2)
Approve execution? (yes / no): yes
Human approved execution.
Result: 25

===== Request: create a file called file.txt and open it for writing =====
Expression parsed: open('file.txt', 'w')
Approve execution? (yes / no): yes
Human approved execution.
Result: could not evaluate (NameError: name 'open' is not defined)
